# CircleKit Getting Started (Gateway + x402)

This notebook is a gentle, end-to-end introduction to the SDK:

1. Create or select a Circle developer-controlled wallet
2. Integrate that wallet with `circlekit`
3. Deposit USDC into Gateway
4. Pay an x402-protected endpoint through Gateway
5. Withdraw funds from Gateway

## What is Gateway in this context?

Gateway is the payment rail used by Circle x402 batching. In practice:

- `deposit(...)` moves funds from your wallet into your Gateway balance
- `pay(...)` spends from Gateway balance to pay x402 endpoints
- `withdraw(...)` moves funds back out from Gateway to wallet

## Prerequisites

- Run from repo root
- Install dependencies:

```bash
uv sync --extra wallets
```


## Start the packaged seller server

In a separate terminal, run the server shipped with this repo:

```bash
SELLER_ADDRESS=0xYOUR_SELLER_ADDRESS uv run --with fastapi --with uvicorn python examples/paywall_server.py
```

This server exposes:

- `GET /health` (free)
- `GET /api/analyze` (\$0.01)
- `POST /api/generate` (\$0.05)


## Fund your wallet (Arc Testnet)

1. Open https://faucet.circle.com/
2. Select **Arc Testnet**
3. Paste your wallet address
4. Request funds

You need test USDC in the buyer wallet before deposit/payment.

In [ ]:
import uuid

import requests
from circle.web3 import utils
from circle.web3.developer_controlled_wallets import (
    CreateWalletRequest,
    CreateWalletSetRequest,
    WalletMetadata,
    WalletsApi,
    WalletSetsApi,
)
from circlekit import GatewayClientSync
from circlekit.wallets import CircleTxExecutor, CircleWalletSigner

## Entity Secret Setup (important)

The `entity_secret` is required for Circle developer-controlled wallet signing and transactions.

One-time registration flow:

1. Generate a 32-byte hex `entity_secret`
2. Register it with Circle using your API key
3. Save the value securely

After registration, reuse the same `entity_secret` in this notebook.

In [ ]:
# Optional helper: generate a new entity secret (print-only)
# Copy the printed value and keep it safe.
# utils.generate_entity_secret()

In [ ]:
# Optional helper: one-time registration (uncomment to run once)
# api_key_for_registration = "TEST_API_KEY:...:..."
# entity_secret_for_registration = "<64-char-hex>"
# utils.register_entity_secret_ciphertext(
#     api_key=api_key_for_registration,
#     entity_secret=entity_secret_for_registration,
#     base_url="https://api.circle.com",  # sandbox: https://api-sandbox.circle.com
# )

## Configure credentials and server

In [ ]:
# Circle credentials
HOST = "https://api.circle.com"  # sandbox: https://api-sandbox.circle.com
api_key = "TEST_API_KEY:...:..."
entity_secret = "<64-char-hex-entity-secret>"

SERVER_URL = "http://127.0.0.1:4022"
PAID_URL = f"{SERVER_URL}/api/analyze"

requests.get(f"{SERVER_URL}/health", timeout=10).json()

## Initialize Circle developer-controlled wallet client

In [ ]:
dcw_client = utils.init_developer_controlled_wallets_client(
    api_key=api_key,
    entity_secret=entity_secret,
    host=HOST,
)
wallet_sets_api = WalletSetsApi(dcw_client)
wallets_api = WalletsApi(dcw_client)

## Option A: Create a wallet set + wallet (one-time)

If you already have wallet id/address, skip to Option B.

In [ ]:
ws = wallet_sets_api.create_wallet_set(
    CreateWalletSetRequest(
        idempotencyKey=str(uuid.uuid4()),
        name="circlekit-getting-started",
    )
)
wallet_set_id = ws.data.wallet_set.actual_instance.id
wallet_set_id

In [ ]:
wr = wallets_api.create_wallet(
    CreateWalletRequest(
        idempotencyKey=str(uuid.uuid4()),
        walletSetId=wallet_set_id,
        blockchains=["ARC-TESTNET"],
        count=1,
        metadata=[WalletMetadata(name="circlekit-buyer")],
    )
)
wallet = getattr(wr.data.wallets[0], "actual_instance", wr.data.wallets[0])
buyer_wallet_id = wallet.id
buyer_wallet_address = wallet.address
buyer_wallet_id, buyer_wallet_address

## Option B: Reuse existing wallet

If you already have these values, set them directly.

In [ ]:
# Uncomment and set if reusing existing buyer wallet
# buyer_wallet_id = "<existing-arc-wallet-id>"
# buyer_wallet_address = "0x..."

buyer_wallet_id, buyer_wallet_address

## Integrate Circle wallet with circlekit

In [ ]:
signer = CircleWalletSigner(
    wallet_id=buyer_wallet_id,
    wallet_address=buyer_wallet_address,
    api_key=api_key,
    entity_secret=entity_secret,
)

tx_executor = CircleTxExecutor(
    wallet_id=buyer_wallet_id,
    wallet_address=buyer_wallet_address,
    api_key=api_key,
    entity_secret=entity_secret,
)

client = GatewayClientSync(chain="arcTestnet", signer=signer, tx_executor=tx_executor)
client.address

## Check endpoint support + balances

In [ ]:
supports = client.supports(PAID_URL)
supports

In [ ]:
balances = client.get_balances()
{
    "wallet_usdc": balances.wallet.formatted,
    "gateway_available": balances.gateway.formatted_available,
    "gateway_withdrawing": balances.gateway.formatted_withdrawing,
    "gateway_withdrawable": balances.gateway.formatted_withdrawable,
}

## Deposit to Gateway

This funds your Gateway balance so `pay(...)` can spend from it.

In [ ]:
dep = client.deposit("1.0")
dep

## Make an x402 payment via Gateway

In [ ]:
pay = client.pay(PAID_URL)
pay

## Withdraw from Gateway back to wallet

In [ ]:
wd = client.withdraw("0.15")
wd

## Optional: Trustless withdrawal path

This is a delayed two-step path:

1. `initiate_trustless_withdrawal(...)`
2. Wait until eligible block
3. `complete_trustless_withdrawal()`

Depending on chain settings, wait time can be long.

In [ ]:
delay = client.get_trustless_withdrawal_delay()
tw_init = client.initiate_trustless_withdrawal("0.10")
delay, tw_init

In [ ]:
try:
    tw_done = client.complete_trustless_withdrawal()
    print(tw_done)
except Exception as e:
    print(type(e).__name__, str(e))

## Cleanup

In [ ]:
client.close()